In [1]:
import influxdb
from pprint import pprint

In [2]:
client = influxdb.InfluxDBClient()
#client.create_database('ProfAll')

In [3]:
client.switch_database('ProfAll')
client.get_list_measurements()

[{'name': 'invocation'}, {'name': 'profile'}, {'name': 'runtime'}]

In [25]:
client.create_retention_policy('default', replication=1, duration='7d', default=True)

In [23]:
client.drop_retention_policy('default')

In [4]:
for invocation in list(client.query('SELECT * FROM runtime'))[0]:
    print(f"{invocation}\n")
      

{'time': '2019-08-26T18:08:21.847897014Z', 'args': 'test.py', 'interpretter': '/usr/bin/python', 'pid': '24808', 'ppid': '24692', 'runtime': 0.005945, 'version': '2.7.15.final.0'}

{'time': '2019-08-26T18:08:45.123901457Z', 'args': '/usr/share/unattended-upgrades/unattended-upgrade-shutdown --wait-for-signal', 'interpretter': '/usr/bin/python3', 'pid': '1324', 'ppid': '1', 'runtime': 2616.801174, 'version': '3.6.8.final.0'}

{'time': '2019-08-26T18:11:07.94739475Z', 'args': '/usr/lib/x86_64-linux-gnu/netdata/plugins.d/python.d.plugin 1', 'interpretter': '/usr/bin/python', 'pid': '2847', 'ppid': '2622', 'runtime': 0.16309, 'version': '2.7.15.final.0'}

{'time': '2019-08-26T18:23:58.546873682Z', 'args': '/usr/local/bin/jupyter-notebook list', 'interpretter': '/usr/bin/python3', 'pid': '9670', 'ppid': '9625', 'runtime': 1.0925, 'version': '3.6.8.final.0'}

{'time': '2019-08-26T18:24:19.809325773Z', 'args': '/usr/bin/dropbox start -i', 'interpretter': '/usr/bin/python3', 'pid': '10181', 'p

In [5]:
summary = set()
for invocation in list(client.query("SELECT * FROM runtime WHERE interpreter!='/home/android/.virtualenvs/ProfAll/bin/python'"))[0]:
    print(f"{invocation}\n")
    summary.add((invocation['interpretter'],invocation['args']))
      
pprint(summary)

{'time': '2019-08-26T18:08:21.847897014Z', 'args': 'test.py', 'interpretter': '/usr/bin/python', 'pid': '24808', 'ppid': '24692', 'runtime': 0.005945, 'version': '2.7.15.final.0'}

{'time': '2019-08-26T18:08:45.123901457Z', 'args': '/usr/share/unattended-upgrades/unattended-upgrade-shutdown --wait-for-signal', 'interpretter': '/usr/bin/python3', 'pid': '1324', 'ppid': '1', 'runtime': 2616.801174, 'version': '3.6.8.final.0'}

{'time': '2019-08-26T18:11:07.94739475Z', 'args': '/usr/lib/x86_64-linux-gnu/netdata/plugins.d/python.d.plugin 1', 'interpretter': '/usr/bin/python', 'pid': '2847', 'ppid': '2622', 'runtime': 0.16309, 'version': '2.7.15.final.0'}

{'time': '2019-08-26T18:23:58.546873682Z', 'args': '/usr/local/bin/jupyter-notebook list', 'interpretter': '/usr/bin/python3', 'pid': '9670', 'ppid': '9625', 'runtime': 1.0925, 'version': '3.6.8.final.0'}

{'time': '2019-08-26T18:24:19.809325773Z', 'args': '/usr/bin/dropbox start -i', 'interpretter': '/usr/bin/python3', 'pid': '10181', 'p

In [17]:
for invocation in list(client.query("SELECT * FROM runtime WHERE interpreter!='/home/android/.virtualenvs/ProfAll/bin/python' AND interpreter!='/usr/bin/python3'"))[0]:
    print(f"{invocation}\n")
      

{'time': '2019-08-26T02:36:04.571177146Z', 'args': 'test.py', 'interpretter': '/usr/bin/python', 'pid': '30618', 'ppid': '7505', 'runtime': 0.961395, 'version': '2.7.15.final.0'}



In [ ]:
for invocation in list(client.query('SELECT * FROM profile'))[0]:
    print(f"{invocation}\n")
    
 

In [6]:
from distutils import sysconfig
import os.path
site_packages_path = sysconfig.get_python_lib()
site_packages_path

'/home/android/.virtualenvs/ProfAll/lib/python3.7/site-packages'

In [7]:
os.path.join(*site_packages_path.split(os.sep)[-3:])

'lib/python3.7/site-packages'

In [8]:
import site
import os.path
site_packages_path = site.getsitepackages()[0]
site_packages_path

'/home/android/.virtualenvs/ProfAll/lib/python3.7/site-packages'

In [9]:
os.path.join(*site_packages_path.split(os.sep)[-3:])

'lib/python3.7/site-packages'

# Injecting Metric Collection into Applications without Changing the Code

## Description 

While we usually collect metrics by writing the collection into our code, there are cases where we would like to get metrics from all the scripts on the system without editing all of them--perhaps without even knowing what's running.  This can solve problems like:

* tracking interpreter usage by version 
* standardizing metrics across applications 
* profiling applications across an entire machine

This talk includes techniques based around:

* site.py
* sitecustomize & usercustomize
* .pth files

## Audience Take-Aways

The audience will learn how to use site.py and related files for metrics gathering and to otherwise participate in the interpreter startup process. 

## Outline

* Intro (2 minutes)
  - Andy Intro
  - Bloomberg Intro 
* Metrics in our direct control (4 minutes)
  - logs 
  - service metrics
  - machine metrics
  - tools: top, ps, prometheus/node_exporter, Splunk, Elk Stack, etc
* But sometimes we don't intend to edit all our code (2 minutes)
  - examples finding legacy Python 2...3.7 usage, module usage, basic benchmarking
* Interfacing with the interpreter startup (3 minutes)
  - site.py
    * sitecustomize & usercustomize
  - .pth files
  - risks and gotchas
* Example: Reporting Executions (4 minutes)
* Example: Profiling (4 minutes)
* Installation How to and Considerations (4 minutes)
* Conclusion (2 minutes)
* Q & A (5 minutes)

X  I accept the PyCon Canada 2019 recording release

  - "Standard" Profiling
     * What is this program doing?
     * When it causes problems what is it doing?
     * Could we change that to eliminate our problems without creating new ones?
     * tools: cprofile, valgrind, vtune, etc
  - "Mass" Profiling aka Metrics
     * What is the system (or cluster) doing?
     * Where is problematic or undesirable software running?
     * Which applications need further attention?